# CustomerIQ: Data Exploration and Analysis

This notebook provides comprehensive data exploration for the CustomerIQ platform - an intelligent customer segmentation and churn prevention system.

## Objectives:
- Load and explore customer transaction data
- Assess data quality and identify issues
- Perform exploratory data analysis with visualizations
- Prepare data for RFM analysis and machine learning models

## 1. Environment Setup and Library Imports

In [ ]:
# Core data manipulation and analysis
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine learning libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Database connectivity
from sqlalchemy import create_engine
import sqlite3

# Statistical analysis
from scipy import stats
import statsmodels.api as sm

# System and path utilities
import sys
import os
from pathlib import Path

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('default')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn available: {True}")

## 2. Data Loading and Initial Setup

In [ ]:
# Add project source directory to path
project_root = Path.cwd().parent
src_path = project_root / 'src'
sys.path.append(str(src_path))

# Import custom modules
try:
    from data_processor import DataProcessor
    from rfm_analyzer import RFMAnalyzer
    from churn_predictor import ChurnPredictor
    from database import DatabaseManager
    print("✅ Custom modules imported successfully!")
except ImportError as e:
    print(f"❌ Error importing custom modules: {e}")
    print("Using standalone analysis approach...")

# Initialize data processor
data_processor = DataProcessor()

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Create sample dataset for demonstration
print("Creating sample e-commerce dataset...")

# Generate sample transaction data
sample_data = data_processor.create_sample_dataset(n_customers=1000, n_transactions_range=(1, 50))

print(f"Sample dataset created with {len(sample_data)} transactions")
print(f"Unique customers: {sample_data['customer_id'].nunique()}")
print(f"Date range: {sample_data['order_date'].min()} to {sample_data['order_date'].max()}")

# Display first few rows
sample_data.head()

## 3. Basic Data Information and Statistics

In [ ]:
# Basic dataset information
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Dataset Shape: {sample_data.shape}")
print(f"Memory Usage: {sample_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Number of Customers: {sample_data['customer_id'].nunique():,}")
print(f"Number of Transactions: {len(sample_data):,}")
print(f"Date Range: {(sample_data['order_date'].max() - sample_data['order_date'].min()).days} days")

print("\n📋 COLUMN INFORMATION")
print("=" * 50)
sample_data.info()

In [ ]:
# Descriptive statistics
print("📈 DESCRIPTIVE STATISTICS")
print("=" * 50)

# Numeric columns statistics
numeric_stats = sample_data.describe()
display(numeric_stats)

print("\n📊 CATEGORICAL VARIABLES")
print("=" * 50)

# Product category distribution
if 'product_category' in sample_data.columns:
    category_dist = sample_data['product_category'].value_counts()
    print("Product Category Distribution:")
    for category, count in category_dist.items():
        percentage = (count / len(sample_data)) * 100
        print(f"  {category}: {count:,} ({percentage:.1f}%)")

## 4. Data Quality Assessment

In [ ]:
# Check for missing values
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 50)

missing_data = sample_data.isnull().sum()
missing_percentage = (missing_data / len(sample_data)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
})

print("Missing Values Summary:")
display(missing_summary[missing_summary['Missing Count'] > 0])

if missing_summary['Missing Count'].sum() == 0:
    print("✅ No missing values found!")

# Check for duplicates
duplicates = sample_data.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Check data types
print("\nData Types:")
print(sample_data.dtypes)

In [ ]:
# Outlier detection for order values
print("🚨 OUTLIER DETECTION")
print("=" * 50)

# Calculate outliers using IQR method
Q1 = sample_data['order_value'].quantile(0.25)
Q3 = sample_data['order_value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = sample_data[(sample_data['order_value'] < lower_bound) | 
                      (sample_data['order_value'] > upper_bound)]

print(f"Order Value Statistics:")
print(f"  Q1 (25th percentile): ${Q1:.2f}")
print(f"  Q3 (75th percentile): ${Q3:.2f}")
print(f"  IQR: ${IQR:.2f}")
print(f"  Lower bound: ${lower_bound:.2f}")
print(f"  Upper bound: ${upper_bound:.2f}")
print(f"  Outliers found: {len(outliers)} ({len(outliers)/len(sample_data)*100:.2f}%)")

if len(outliers) > 0:
    print("\nOutlier Summary:")
    print(f"  Minimum outlier value: ${outliers['order_value'].min():.2f}")
    print(f"  Maximum outlier value: ${outliers['order_value'].max():.2f}")

## 5. Exploratory Data Analysis with Visualizations

In [ ]:
# Order value distribution
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Order Value Distribution', 'Order Value Box Plot', 
                   'Log-transformed Order Values', 'Order Value by Category'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Histogram
fig.add_trace(
    go.Histogram(x=sample_data['order_value'], nbinsx=50, name='Order Value'),
    row=1, col=1
)

# Box plot
fig.add_trace(
    go.Box(y=sample_data['order_value'], name='Order Value'),
    row=1, col=2
)

# Log-transformed values
log_values = np.log1p(sample_data['order_value'])
fig.add_trace(
    go.Histogram(x=log_values, nbinsx=50, name='Log Order Value'),
    row=2, col=1
)

# Order value by category
if 'product_category' in sample_data.columns:
    category_values = sample_data.groupby('product_category')['order_value'].mean().sort_values()
    fig.add_trace(
        go.Bar(x=category_values.index, y=category_values.values, name='Avg Order Value'),
        row=2, col=2
    )

fig.update_layout(height=600, showlegend=False, title_text="Order Value Analysis")
fig.show()

In [ ]:
# Temporal analysis
sample_data['order_date'] = pd.to_datetime(sample_data['order_date'])
sample_data['year_month'] = sample_data['order_date'].dt.to_period('M')
sample_data['day_of_week'] = sample_data['order_date'].dt.day_name()
sample_data['hour'] = sample_data['order_date'].dt.hour

# Monthly trends
monthly_trends = sample_data.groupby('year_month').agg({
    'order_value': ['sum', 'count', 'mean'],
    'customer_id': 'nunique'
}).round(2)

monthly_trends.columns = ['Total Revenue', 'Total Orders', 'Avg Order Value', 'Unique Customers']
monthly_trends = monthly_trends.reset_index()
monthly_trends['year_month'] = monthly_trends['year_month'].astype(str)

# Create temporal visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend', 'Day of Week Analysis', 
                   'Monthly Order Count', 'Average Order Value Trend')
)

# Monthly revenue
fig.add_trace(
    go.Scatter(x=monthly_trends['year_month'], y=monthly_trends['Total Revenue'], 
              mode='lines+markers', name='Revenue'),
    row=1, col=1
)

# Day of week analysis
dow_analysis = sample_data.groupby('day_of_week')['order_value'].agg(['count', 'sum']).reset_index()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_analysis['day_of_week'] = pd.Categorical(dow_analysis['day_of_week'], categories=day_order, ordered=True)
dow_analysis = dow_analysis.sort_values('day_of_week')

fig.add_trace(
    go.Bar(x=dow_analysis['day_of_week'], y=dow_analysis['count'], name='Orders by Day'),
    row=1, col=2
)

# Monthly order count
fig.add_trace(
    go.Scatter(x=monthly_trends['year_month'], y=monthly_trends['Total Orders'], 
              mode='lines+markers', name='Orders'),
    row=2, col=1
)

# Average order value trend
fig.add_trace(
    go.Scatter(x=monthly_trends['year_month'], y=monthly_trends['Avg Order Value'], 
              mode='lines+markers', name='AOV'),
    row=2, col=2
)

fig.update_layout(height=600, showlegend=False, title_text="Temporal Analysis")
fig.show()

## 6. Customer-Level Analysis

In [ ]:
# Create customer-level features
print("👥 CUSTOMER-LEVEL ANALYSIS")
print("=" * 50)

# Aggregate data by customer
customer_summary = sample_data.groupby('customer_id').agg({
    'order_date': ['min', 'max', 'count'],
    'order_value': ['sum', 'mean', 'std'],
    'quantity': ['sum', 'mean'] if 'quantity' in sample_data.columns else ['count'] * 2,
    'product_category': 'nunique' if 'product_category' in sample_data.columns else 'count'
}).round(2)

# Flatten column names
customer_summary.columns = [
    'first_purchase', 'last_purchase', 'total_orders',
    'total_spent', 'avg_order_value', 'std_order_value',
    'total_quantity', 'avg_quantity',
    'category_diversity'
]

# Calculate additional features
customer_summary['customer_lifespan'] = (
    customer_summary['last_purchase'] - customer_summary['first_purchase']
).dt.days

reference_date = sample_data['order_date'].max()
customer_summary['days_since_last_purchase'] = (
    reference_date - customer_summary['last_purchase']
).dt.days

# Handle missing std values (single purchase customers)
customer_summary['std_order_value'] = customer_summary['std_order_value'].fillna(0)

# Display customer summary statistics
print(f"Customer Summary Statistics:")
print(f"Total Customers: {len(customer_summary):,}")
print(f"Avg Orders per Customer: {customer_summary['total_orders'].mean():.1f}")
print(f"Avg Spending per Customer: ${customer_summary['total_spent'].mean():.2f}")
print(f"Avg Customer Lifespan: {customer_summary['customer_lifespan'].mean():.0f} days")

customer_summary.head()

In [ ]:
# Customer behavior analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Customer Order Frequency', 'Customer Spending Distribution', 
                   'Customer Lifespan', 'Days Since Last Purchase')
)

# Order frequency distribution
fig.add_trace(
    go.Histogram(x=customer_summary['total_orders'], nbinsx=20, name='Order Frequency'),
    row=1, col=1
)

# Spending distribution
fig.add_trace(
    go.Histogram(x=customer_summary['total_spent'], nbinsx=30, name='Total Spending'),
    row=1, col=2
)

# Customer lifespan
fig.add_trace(
    go.Histogram(x=customer_summary['customer_lifespan'], nbinsx=25, name='Lifespan (days)'),
    row=2, col=1
)

# Days since last purchase
fig.add_trace(
    go.Histogram(x=customer_summary['days_since_last_purchase'], nbinsx=30, name='Recency (days)'),
    row=2, col=2
)

fig.update_layout(height=600, showlegend=False, title_text="Customer Behavior Analysis")
fig.show()

## 7. Correlation Analysis and Feature Relationships

In [ ]:
# Correlation analysis
print("🔗 CORRELATION ANALYSIS")
print("=" * 50)

# Select numeric columns for correlation
numeric_columns = [
    'total_orders', 'total_spent', 'avg_order_value', 'std_order_value',
    'total_quantity', 'avg_quantity', 'category_diversity',
    'customer_lifespan', 'days_since_last_purchase'
]

correlation_matrix = customer_summary[numeric_columns].corr()

# Create correlation heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.around(correlation_matrix.values, decimals=2),
    texttemplate="%{text}",
    textfont={"size": 10},
    showscale=True
))

fig.update_layout(
    title="Customer Features Correlation Matrix",
    width=800,
    height=600
)

fig.show()

# Identify strongest correlations
print("\nStrongest Correlations (absolute value > 0.5):")
strong_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            strong_corr.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlation': corr_value
            })

if strong_corr:
    strong_corr_df = pd.DataFrame(strong_corr).sort_values('Correlation', key=abs, ascending=False)
    display(strong_corr_df)
else:
    print("No correlations with absolute value > 0.5 found.")

## 8. Advanced Customer Segmentation Preview

In [ ]:
# Prepare data for RFM analysis preview
print("🎯 RFM ANALYSIS PREVIEW")
print("=" * 50)

# Calculate RFM components
rfm_data = customer_summary.copy()
rfm_data['Recency'] = rfm_data['days_since_last_purchase']
rfm_data['Frequency'] = rfm_data['total_orders']
rfm_data['Monetary'] = rfm_data['total_spent']

# Calculate RFM scores using quintiles
rfm_data['R_Score'] = pd.qcut(rfm_data['Recency'], q=5, labels=[5, 4, 3, 2, 1])  # Lower recency = higher score
rfm_data['F_Score'] = pd.qcut(rfm_data['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm_data['M_Score'] = pd.qcut(rfm_data['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Convert to numeric
rfm_data['R_Score'] = rfm_data['R_Score'].astype(int)
rfm_data['F_Score'] = rfm_data['F_Score'].astype(int)
rfm_data['M_Score'] = rfm_data['M_Score'].astype(int)

# Create RFM combined score
rfm_data['RFM_Score'] = (
    rfm_data['R_Score'].astype(str) + 
    rfm_data['F_Score'].astype(str) + 
    rfm_data['M_Score'].astype(str)
)

# Simple segmentation based on RFM scores
def simple_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 4 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Cannot Lose Them'
    elif r <= 3 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Others'

rfm_data['Segment'] = rfm_data.apply(simple_segment, axis=1)

# Display segment distribution
segment_dist = rfm_data['Segment'].value_counts()
print("Customer Segment Distribution:")
for segment, count in segment_dist.items():
    percentage = (count / len(rfm_data)) * 100
    print(f"  {segment}: {count:,} ({percentage:.1f}%)")

# Visualize segment distribution
fig = px.pie(
    values=segment_dist.values,
    names=segment_dist.index,
    title="Customer Segment Distribution (RFM Analysis)"
)
fig.show()

rfm_data[['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment']].head(10)

## 9. Data Export and Summary

In [ ]:
# Generate final data quality report
print("📋 FINAL DATA QUALITY REPORT")
print("=" * 50)

quality_report = {
    'Dataset Overview': {
        'Total Transactions': len(sample_data),
        'Unique Customers': sample_data['customer_id'].nunique(),
        'Date Range (days)': (sample_data['order_date'].max() - sample_data['order_date'].min()).days,
        'Total Revenue': f"${sample_data['order_value'].sum():,.2f}",
        'Average Order Value': f"${sample_data['order_value'].mean():.2f}"
    },
    'Data Quality': {
        'Missing Values': sample_data.isnull().sum().sum(),
        'Duplicate Rows': sample_data.duplicated().sum(),
        'Outliers (Order Value)': len(outliers),
        'Data Completeness': f"{((len(sample_data) - sample_data.isnull().sum().sum()) / (len(sample_data) * len(sample_data.columns))) * 100:.1f}%"
    },
    'Customer Insights': {
        'Avg Orders per Customer': f"{customer_summary['total_orders'].mean():.1f}",
        'Avg Spending per Customer': f"${customer_summary['total_spent'].mean():.2f}",
        'Avg Customer Lifespan': f"{customer_summary['customer_lifespan'].mean():.0f} days",
        'Single Purchase Customers': f"{(customer_summary['total_orders'] == 1).sum()} ({(customer_summary['total_orders'] == 1).mean() * 100:.1f}%)"
    }
}

for category, metrics in quality_report.items():
    print(f"\n{category}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value}")

# Save processed data
print("\n💾 SAVING PROCESSED DATA")
print("=" * 50)

# Create data directory if it doesn't exist
data_dir = project_root / 'data' / 'processed'
data_dir.mkdir(parents=True, exist_ok=True)

# Save datasets
transaction_file = data_dir / 'clean_transactions.csv'
customer_file = data_dir / 'customer_features.csv'
rfm_file = data_dir / 'rfm_analysis.csv'

sample_data.to_csv(transaction_file, index=False)
customer_summary.to_csv(customer_file, index=True)
rfm_data.to_csv(rfm_file, index=True)

print(f"✅ Transaction data saved to: {transaction_file}")
print(f"✅ Customer features saved to: {customer_file}")
print(f"✅ RFM analysis saved to: {rfm_file}")

print("\n🎉 Data exploration completed successfully!")
print("Next steps: Proceed to RFM Analysis and Churn Prediction notebooks.")